# WPM Visual Explainer

This notebook is a CPU-quick visual explanation of the Wave Propagation Method (WPM) used in this repository.

The explainer now uses the same sample identity as the paper examples: an Au [100] FCC cell with lattice constant `a = 4.076 A`. To keep the notebook lightweight and runnable without abTEM/CuPy potential generation, the transverse slice is a small analytic Au [100] atom-column potential drawn directly from the FCC unit-cell geometry.

The potential is optionally amplified by `potential_visual_gain` so that the WPM-vs-ASM mechanism remains visible on a small grid. The geometry is gold; the gain is only for visual pedagogy.

- Angular spectrum multislice applies a local phase screen, then propagates through vacuum with the exact free-space dispersion.
- WPM treats each slice as locally homogeneous and propagates with a local refractive index.
- The WPM advantage becomes visible when lateral refractive-index variation, finite slice thickness, and high-angle components are all present.


In [ ]:
import os
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "wide_angle_propagation").exists():
        ROOT = candidate
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

try:
    from wide_angle_propagation.propagation_methods import (
        Propagator,
        angular_spectrum_propagation_kernel,
        electron_refractive_index,
        energy2wavelength,
        wpm_step_adaptive,
    )
    USING_PACKAGE_WPM = True
except ModuleNotFoundError as exc:
    print(f"Using local WPM fallback because a package dependency is missing: {exc.name}")
    USING_PACKAGE_WPM = False

    ELECTRON_REST_ENERGY_EV = 510_998.95000
    PLANCK = 6.62607015e-34
    LIGHT_SPEED = 299_792_458.0
    ELECTRON_CHARGE = 1.602176634e-19

    def energy2wavelength(energy):
        energy = jnp.asarray(energy, dtype=jnp.float64)
        return PLANCK * LIGHT_SPEED / (ELECTRON_CHARGE * jnp.sqrt(energy * (2.0 * ELECTRON_REST_ENERGY_EV + energy))) * 1.0e10

    def electron_refractive_index(potential, energy):
        potential = jnp.asarray(potential, dtype=jnp.float64)
        energy = jnp.asarray(energy, dtype=jnp.float64)
        e_minus_v = energy + potential
        numerator = 2.0 * e_minus_v * ELECTRON_REST_ENERGY_EV + e_minus_v**2
        denominator = 2.0 * energy * ELECTRON_REST_ENERGY_EV + energy**2
        return jnp.sqrt(numerator / denominator)

    def get_frequencies(n, m, ps):
        fy = jnp.fft.fftfreq(n, ps[0])
        fx = jnp.fft.fftfreq(m, ps[1])
        return jnp.meshgrid(fy, fx, indexing="ij")

    def transverse_wavenumber_squared(shape, sampling):
        fy, fx = get_frequencies(shape[0], shape[1], sampling)
        return (2.0 * jnp.pi * fy) ** 2 + (2.0 * jnp.pi * fx) ** 2

    def angular_spectrum_propagation_kernel(n, m, ps, z, energy):
        wavelength = energy2wavelength(energy)
        fy, fx = get_frequencies(n, m, ps)
        kz_sq = (1.0 / wavelength) ** 2 - fx**2 - fy**2
        kz = jnp.sqrt(jnp.asarray(kz_sq, dtype=jnp.complex128))
        return jnp.exp(1j * 2.0 * jnp.pi * z * kz)

    def Propagator(u, H):
        return jnp.fft.ifft2(H * jnp.fft.fft2(u))

    def smoothstep(x):
        x = jnp.clip(x, 0.0, 1.0)
        return 3.0 * x**2 - 2.0 * x**3

    def get_polynomial_bins(n_min, n_max, n_bins, power=2.0):
        t = jnp.linspace(0.0, 1.0, n_bins)
        return n_min + (n_max - n_min) * t**power

    def wpm_propagation_kernel(Ek, n_val, k0, k_perp2, dz):
        kz = jnp.sqrt(jnp.asarray(n_val**2 * k0**2 - k_perp2, dtype=jnp.complex128))
        H = jnp.exp(1j * dz * kz)
        return jnp.fft.ifft2(H * Ek)

    wpm_propagation_kernel_vmap = jax.vmap(
        wpm_propagation_kernel,
        in_axes=(None, 0, None, None, None),
    )

    def wpm_step_adaptive(wave, n_map, dz, energy, ps, n_bins=256, power_spacing=2.0):
        ny, nx = wave.shape
        wavelength = energy2wavelength(energy)
        k0 = 2.0 * jnp.pi / wavelength
        k_perp2 = transverse_wavenumber_squared((ny, nx), ps)
        Ek = jnp.fft.fft2(wave)

        n_min, n_max = n_map.min(), n_map.max()
        n_refs = get_polynomial_bins(n_min, n_max, n_bins, power=power_spacing)
        ref_fields = wpm_propagation_kernel_vmap(Ek, n_refs, k0, k_perp2, dz)

        idx_R = jnp.searchsorted(n_refs, n_map)
        idx_R = jnp.clip(idx_R, 1, n_bins - 1)
        idx_L = idx_R - 1

        n_L = n_refs[idx_L]
        n_R = n_refs[idx_R]
        denom = n_R - n_L
        w_raw = (n_map - n_L) / jnp.where(denom == 0, 1.0, denom)
        w = smoothstep(w_raw)

        field_L = jnp.take_along_axis(ref_fields, idx_L[None, ...], axis=0).squeeze()
        field_R = jnp.take_along_axis(ref_fields, idx_R[None, ...], axis=0).squeeze()
        return (1.0 - w) * field_L + w * field_R, w, idx_L, n_refs

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": False,
    "image.origin": "lower",
})

print(f"Repository root: {ROOT}")
print(f"JAX backend: {jax.default_backend()}")
print(f"WPM helpers: {'package implementation' if USING_PACKAGE_WPM else 'local notebook fallback'}")


## Configuration

The default is a small Au [100] supercell with the same lattice constant used in the Au benchmarks. The optical strength is deliberately adjustable for the explainer: `potential_visual_gain = 1` is the unamplified analytic potential scale, while larger values make the phase accumulation easier to see on a compact CPU grid.


In [ ]:
shape = (128, 128)

au_lattice_A = 4.076
au_repeats_xy = (4, 4)  # (x, y) repeats of the cubic Au [100] unit cell
fov_x_A = au_repeats_xy[0] * au_lattice_A
fov_y_A = au_repeats_xy[1] * au_lattice_A
sampling = (fov_y_A / shape[0], fov_x_A / shape[1])  # (dy, dx), Angstrom per pixel

energy = 200_000.0
n_slices_per_cell = 12
dz = au_lattice_A / n_slices_per_cell
n_slices = n_slices_per_cell

atom_sigma_A = 0.20
peak_potential_V = 220.0
potential_visual_gain = 35.0
n_bins = 8

probe_waist_A = 2.4
probe_tilt_x_mrad = 70.0

wavelength = float(energy2wavelength(energy))
print(f"Au [100] visual supercell: {au_repeats_xy[0]} x {au_repeats_xy[1]} cells, a = {au_lattice_A:.3f} A")
print(f"lambda = {wavelength:.5f} A at {energy/1e3:.0f} keV")
print(f"grid = {shape}, field of view = {fov_x_A:.2f} x {fov_y_A:.2f} A")
print(f"sampling (dy, dx) = ({sampling[0]:.4f}, {sampling[1]:.4f}) A")
print(f"dz = {dz:.4f} A, n_bins = {n_bins}, visual potential gain = {potential_visual_gain:g}")


## Helper Functions

These helpers keep the notebook focused on the mechanism. The WPM propagation itself still calls the package implementation.


In [ ]:
def as_numpy(x):
    return np.asarray(jax.device_get(x))


def periodic_delta(values, center, period):
    return (values - center + 0.5 * period) % period - 0.5 * period


def make_fcc_au_100_positions(repeats_xy, lattice_A):
    nx_rep, ny_rep = repeats_xy
    conventional_fcc = np.array([
        [0.0, 0.0, 0.0],
        [0.0, 0.5, 0.5],
        [0.5, 0.0, 0.5],
        [0.5, 0.5, 0.0],
    ])
    positions = []
    for ix in range(nx_rep):
        for iy in range(ny_rep):
            offset = np.array([ix, iy, 0.0])
            positions.extend((conventional_fcc + offset) * lattice_A)
    return np.asarray(positions, dtype=np.float64)


def make_gold_100_sample(
    shape,
    repeats_xy,
    lattice_A,
    atom_sigma_A,
    peak_potential_V,
    potential_visual_gain,
    energy,
):
    atom_positions_A = make_fcc_au_100_positions(repeats_xy, lattice_A)

    ny, nx = shape
    fov_x_A = repeats_xy[0] * lattice_A
    fov_y_A = repeats_xy[1] * lattice_A
    fov_z_A = lattice_A
    dy = fov_y_A / ny
    dx = fov_x_A / nx

    x = (np.arange(nx) + 0.5) * dx
    y = (np.arange(ny) + 0.5) * dy
    X, Y = np.meshgrid(x, y)

    atom_columns = np.zeros(shape, dtype=np.float64)
    for x0, y0, _ in atom_positions_A:
        dX = periodic_delta(X, x0, fov_x_A)
        dY = periodic_delta(Y, y0, fov_y_A)
        atom_columns += np.exp(-(dX**2 + dY**2) / (2.0 * atom_sigma_A**2))

    atom_columns /= atom_columns.max()
    potential_slice_V = peak_potential_V * potential_visual_gain * atom_columns
    n_map = electron_refractive_index(jnp.asarray(potential_slice_V), energy)

    return {
        "atom_positions_A": atom_positions_A,
        "atom_columns": atom_columns,
        "potential_slice_V": potential_slice_V,
        "n_map": n_map,
        "sampling": (float(dy), float(dx)),
        "extent_A": [0.0, float(fov_x_A), 0.0, float(fov_y_A)],
        "x_axis_A": x,
        "y_axis_A": y,
        "cell_lengths_A": (float(fov_x_A), float(fov_y_A), float(fov_z_A)),
    }

def box_blur_periodic(n_map, passes=4):
    arr = as_numpy(n_map).astype(np.float64)
    for _ in range(passes):
        arr = (
            4.0 * arr
            + np.roll(arr, 1, axis=0)
            + np.roll(arr, -1, axis=0)
            + np.roll(arr, 1, axis=1)
            + np.roll(arr, -1, axis=1)
        ) / 8.0
    return jnp.asarray(arr, dtype=jnp.float64)


def make_tilted_gaussian_probe(shape, sampling, energy, waist_A=2.4, tilt_x_mrad=70.0, center_A=None):
    ny, nx = shape
    dy, dx = sampling
    y = (np.arange(ny) - 0.5 * (ny - 1)) * dy
    x = (np.arange(nx) - 0.5 * (nx - 1)) * dx
    X, Y = np.meshgrid(x, y)

    if center_A is None:
        x0, y0 = 0.0, 0.0
    else:
        x0, y0 = center_A

    wavelength = float(energy2wavelength(energy))
    fx0 = np.sin(tilt_x_mrad * 1e-3) / wavelength
    envelope = np.exp(-((X - x0) ** 2 + (Y - y0) ** 2) / (2.0 * waist_A ** 2))
    phase = np.exp(2j * np.pi * fx0 * X)
    probe = envelope * phase
    probe /= np.sqrt(np.sum(np.abs(probe) ** 2))
    return jnp.asarray(probe, dtype=jnp.complex128)


def asm_step_local_n(wave, n_map, dz, energy, sampling):
    ny, nx = wave.shape
    wavelength = energy2wavelength(energy)
    phase_screen = jnp.exp(1j * 2.0 * jnp.pi * (n_map - 1.0) * dz / wavelength)
    H_vacuum = angular_spectrum_propagation_kernel(ny, nx, sampling, z=dz, energy=energy)
    return Propagator(wave * phase_screen, H_vacuum)


def homogeneous_wpm_field(wave, n_ref, dz, energy, sampling):
    ny, nx = wave.shape
    dy, dx = sampling
    wavelength = energy2wavelength(energy)
    k0 = 2.0 * jnp.pi / wavelength

    ky = 2.0 * jnp.pi * jnp.fft.fftfreq(ny, d=dy)
    kx = 2.0 * jnp.pi * jnp.fft.fftfreq(nx, d=dx)
    Kx, Ky = jnp.meshgrid(kx, ky)
    k_perp2 = Kx ** 2 + Ky ** 2

    kz = jnp.sqrt(jnp.asarray(n_ref ** 2 * k0 ** 2 - k_perp2, dtype=jnp.complex128))
    H = jnp.exp(1j * dz * kz)
    return jnp.fft.ifft2(H * jnp.fft.fft2(wave))


def wpm_diagnostic_step(wave, n_map, dz, energy, sampling, n_bins=8, power_spacing=1.0):
    wave_out, w, idx_L, n_refs = wpm_step_adaptive(
        wave, n_map, dz, energy, sampling, n_bins=n_bins, power_spacing=power_spacing
    )
    idx_R = jnp.clip(idx_L + 1, 0, n_refs.size - 1)

    ref_fields = jnp.stack([
        homogeneous_wpm_field(wave, n_ref, dz, energy, sampling)
        for n_ref in list(n_refs)
    ])

    field_L = jnp.take_along_axis(ref_fields, idx_L[None, ...], axis=0).squeeze()
    field_R = jnp.take_along_axis(ref_fields, idx_R[None, ...], axis=0).squeeze()

    return {
        "wave": wave_out,
        "weights": w,
        "idx_L": idx_L,
        "idx_R": idx_R,
        "n_refs": n_refs,
        "n_L": n_refs[idx_L],
        "n_R": n_refs[idx_R],
        "field_L": field_L,
        "field_R": field_R,
        "ref_fields": ref_fields,
    }


def run_stack(n_map, probe, dz, energy, sampling, n_slices, n_bins=8):
    wave_as = probe
    wave_wpm = probe
    as_history = []
    wpm_history = []

    for _ in range(n_slices):
        wave_as = asm_step_local_n(wave_as, n_map, dz, energy, sampling)
        wave_wpm, _, _, _ = wpm_step_adaptive(
            wave_wpm, n_map, dz, energy, sampling, n_bins=n_bins, power_spacing=1.0
        )
        as_history.append(wave_as)
        wpm_history.append(wave_wpm)

    return wave_as, wave_wpm, jnp.stack(as_history), jnp.stack(wpm_history)


def relative_l2(a, b):
    a = as_numpy(a)
    b = as_numpy(b)
    return float(np.linalg.norm(a - b) / (np.linalg.norm(b) + 1e-300))


def align_global_phase(test_wave, reference_wave):
    test = as_numpy(test_wave)
    ref = as_numpy(reference_wave)
    phase = np.angle(np.vdot(ref.ravel(), test.ravel()))
    return test * np.exp(-1j * phase)


def diffraction_intensity(wave):
    arr = as_numpy(wave)
    dp = np.abs(np.fft.fftshift(np.fft.fft2(arr))) ** 2
    return dp / (dp.sum() + 1e-300)


def angle_grid_mrad(shape, sampling, energy):
    ny, nx = shape
    dy, dx = sampling
    wavelength = float(energy2wavelength(energy))
    fy = np.fft.fftshift(np.fft.fftfreq(ny, d=dy))
    fx = np.fft.fftshift(np.fft.fftfreq(nx, d=dx))
    Fx, Fy = np.meshgrid(fx, fy)
    g = np.sqrt(Fx ** 2 + Fy ** 2)
    return 1000.0 * np.arcsin(np.clip(wavelength * g, 0.0, 1.0))


def annular_relative_l2(dp_test, dp_ref, theta_mrad, n_bins=34):
    bins = np.linspace(0.0, np.nanmax(theta_mrad), n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    values = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (theta_mrad >= lo) & (theta_mrad < hi)
        if mask.sum() < 4:
            values.append(np.nan)
        else:
            values.append(
                np.linalg.norm(dp_test[mask] - dp_ref[mask])
                / (np.linalg.norm(dp_ref[mask]) + 1e-300)
            )
    return centers, np.asarray(values)


def add_colorbar(fig, ax, im, label=None):
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if label is not None:
        cbar.set_label(label)
    cbar.ax.tick_params(labelsize=8)
    return cbar


def plot_gold_sample(gold_sample):
    potential = gold_sample["potential_slice_V"]
    n_map_np = as_numpy(gold_sample["n_map"])
    extent_A = gold_sample["extent_A"]
    atom_positions_A = gold_sample["atom_positions_A"]
    x_axis_A = gold_sample["x_axis_A"]
    center_y = potential.shape[0] // 2

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))

    im0 = axes[0].imshow(potential, extent=extent_A, cmap="magma")
    axes[0].scatter(
        atom_positions_A[:, 0], atom_positions_A[:, 1],
        s=18, facecolors="none", edgecolors="white", linewidths=0.7,
    )
    axes[0].set_title("Au [100] visual potential")
    axes[0].set_xlabel("x (A)")
    axes[0].set_ylabel("y (A)")
    add_colorbar(fig, axes[0], im0, "V")

    n_contrast = 1e3 * (n_map_np - 1.0)
    im1 = axes[1].imshow(n_contrast, extent=extent_A, cmap="viridis")
    axes[1].set_title("refractive-index contrast")
    axes[1].set_xlabel("x (A)")
    axes[1].set_yticks([])
    add_colorbar(fig, axes[1], im1, "1e3 x (n - 1)")

    axes[2].plot(x_axis_A, potential[center_y, :], label="potential")
    axes[2].set_title("central Au-column profile")
    axes[2].set_xlabel("x (A)")
    axes[2].set_ylabel("potential (V)")
    axes[2].grid(True, alpha=0.3)

    fig.suptitle("Gold sample used for the WPM explainer", fontsize=12)
    fig.tight_layout()
    return fig


## Build The Au [100] Sample

The sample is a projected Au [100] atom-column slice. It has the same gold lattice identity as the Au benchmark, but uses directly generated FCC positions and a compact Gaussian column model so the visual explainer remains fast and dependency-light.


In [ ]:
gold_sample = make_gold_100_sample(
    shape=shape,
    repeats_xy=au_repeats_xy,
    lattice_A=au_lattice_A,
    atom_sigma_A=atom_sigma_A,
    peak_potential_V=peak_potential_V,
    potential_visual_gain=potential_visual_gain,
    energy=energy,
)

potential_slice_V = gold_sample["potential_slice_V"]
n_map = gold_sample["n_map"]
sampling = gold_sample["sampling"]
probe = make_tilted_gaussian_probe(shape, sampling, energy, probe_waist_A, probe_tilt_x_mrad)

n_np = as_numpy(n_map)
print(f"Au atoms in visual supercell: {len(gold_sample['atom_positions_A'])}")
print(f"cell lengths (x, y, z) = {gold_sample['cell_lengths_A']}")
print(f"potential range = {potential_slice_V.min():.2f} to {potential_slice_V.max():.2f} V")
print(f"n range = {n_np.min():.8f} to {n_np.max():.8f}")

plot_gold_sample(gold_sample)
plt.show()


## 1D Top-Down View

A useful mental picture is: ASM lets the slice act first as a phase screen, then all transverse pixels propagate through the same vacuum kernel. WPM instead evaluates a small bank of homogeneous-medium propagations and blends them according to the local refractive-index map.


In [ ]:
def plot_xz_schematic(n_map, n_slices=6):
    n_map_np = as_numpy(n_map)
    line = n_map_np[n_map_np.shape[0] // 2]
    contrast_line = 1e3 * (line - 1.0)
    contrast_xz = np.tile(contrast_line, (n_slices, 1))

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    titles = [
        "ASM: Au phase screen, then one vacuum propagator",
        "WPM: Au-index propagator bank, then spatial blend",
    ]

    vmin, vmax = float(np.min(contrast_xz)), float(np.max(contrast_xz))
    for ax, title in zip(axes, titles):
        im = ax.imshow(contrast_xz, aspect="auto", cmap="viridis", origin="upper", vmin=vmin, vmax=vmax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x pixel through Au [100] row")
        ax.set_xticks([])
        ax.set_yticks(np.arange(n_slices))
        ax.set_yticklabels([f"slice {i}" for i in range(n_slices)])
        for y in np.arange(0.5, n_slices, 1.0):
            ax.axhline(y, color="white", lw=0.8, ls="--", alpha=0.7)

    for y in range(n_slices):
        axes[0].annotate("", xy=(112, y + 0.35), xytext=(16, y + 0.35), arrowprops=dict(arrowstyle="->", lw=1.2, color="white"))
        axes[0].text(19, y + 0.05, "same H0", color="white", fontsize=8, weight="bold")

    for y in range(n_slices):
        axes[1].annotate("", xy=(112, y + 0.35), xytext=(16, y + 0.35), arrowprops=dict(arrowstyle="->", lw=1.2, color="white"))
        axes[1].text(18, y + 0.05, "blend H(n_j)", color="white", fontsize=8, weight="bold")

    axes[0].set_ylabel("z propagation")
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02, label="1e3 x (n - 1)")
    fig.tight_layout()
    return fig


plot_xz_schematic(n_map, n_slices=6)
plt.show()


## WPM Binning Diagnostics

For a continuous Au potential, WPM builds a small grid of representative refractive indices. Each output pixel uses its two neighboring refractive-index bins and a smooth interpolation weight. Pixels near gold columns select the higher-index propagators; pixels between columns select lower-index propagators.


In [ ]:
def plot_wpm_diagnostics(n_map, diagnostic, title_suffix=""):
    n_map_np = as_numpy(n_map)
    n_L = as_numpy(diagnostic["n_L"])
    n_R = as_numpy(diagnostic["n_R"])
    weights = as_numpy(diagnostic["weights"])
    field_L = as_numpy(diagnostic["field_L"])
    field_R = as_numpy(diagnostic["field_R"])
    wave = as_numpy(diagnostic["wave"])

    n_contrast = 1e3 * (n_map_np - 1.0)
    n_L_contrast = 1e3 * (n_L - 1.0)
    n_R_contrast = 1e3 * (n_R - 1.0)
    n_limits = (float(n_contrast.min()), float(n_contrast.max()))

    fig, axes = plt.subplots(2, 4, figsize=(13.5, 6.2))
    panels = [
        (n_contrast, "Au n(x, y) - 1", "viridis", n_limits, "1e3 scale"),
        (n_L_contrast, "left bin n_L - 1", "viridis", n_limits, "1e3 scale"),
        (n_R_contrast, "right bin n_R - 1", "viridis", n_limits, "1e3 scale"),
        (weights, "blend weight w", "coolwarm", (0.0, 1.0), None),
        (np.angle(field_L), "phase after H(n_L)", "twilight", (-np.pi, np.pi), "rad"),
        (np.angle(field_R), "phase after H(n_R)", "twilight", (-np.pi, np.pi), "rad"),
        (np.angle(wave), "phase after WPM blend", "twilight", (-np.pi, np.pi), "rad"),
        (np.abs(wave), "amplitude after WPM blend", "magma", None, None),
    ]

    for ax, (arr, title, cmap, limits, label) in zip(axes.ravel(), panels):
        kwargs = {}
        if limits is not None:
            kwargs.update(vmin=limits[0], vmax=limits[1])
        im = ax.imshow(arr, cmap=cmap, **kwargs)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        add_colorbar(fig, ax, im, label)

    fig.suptitle(f"One WPM step through the Au [100] sample{title_suffix}", fontsize=12)
    fig.tight_layout()
    return fig


diagnostic = wpm_diagnostic_step(probe, n_map, dz, energy, sampling, n_bins=n_bins, power_spacing=1.0)
n_refs = as_numpy(diagnostic["n_refs"])
weights = as_numpy(diagnostic["weights"])
print("WPM reference bins, reported as 1e3 x (n - 1):")
print(np.array2string(1e3 * (n_refs - 1.0), precision=4))
print(f"weight range: {weights.min():.3f} to {weights.max():.3f}")
plot_wpm_diagnostics(n_map, diagnostic)
plt.show()


## ASM vs WPM After One Au Unit Cell

Now we propagate the same tilted Gaussian wave through the same static Au [100] visual slice for one unit-cell thickness. The two methods only differ in the propagation step: ASM uses a local phase screen followed by the vacuum angular-spectrum kernel, while WPM uses the local-index propagation bank.


In [ ]:
exit_as, exit_wpm, as_history, wpm_history = run_stack(
    n_map, probe, dz, energy, sampling, n_slices=n_slices, n_bins=n_bins
)
jax.block_until_ready(exit_wpm)

as_aligned = align_global_phase(exit_as, exit_wpm)
wpm_np = as_numpy(exit_wpm)
diff_wave = wpm_np - as_aligned
phase_diff = np.angle(wpm_np * np.conj(as_aligned))

print(f"Exit-wave relative L2 difference, global phase aligned: {np.linalg.norm(diff_wave) / np.linalg.norm(wpm_np):.4e}")


In [ ]:
def plot_exit_comparison(exit_as, exit_wpm):
    as_aligned = align_global_phase(exit_as, exit_wpm)
    wpm = as_numpy(exit_wpm)
    diff = wpm - as_aligned
    phase_diff = np.angle(wpm * np.conj(as_aligned))

    amp_lim = max(np.percentile(np.abs(wpm), 99.5), np.percentile(np.abs(as_aligned), 99.5))
    diff_lim = max(np.percentile(np.abs(diff), 99.5), 1e-14)

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    panels = [
        (np.abs(as_aligned), "ASM exit amplitude", "magma", (0.0, amp_lim)),
        (np.abs(wpm), "WPM exit amplitude", "magma", (0.0, amp_lim)),
        (np.abs(diff), "abs(WPM - ASM)", "inferno", (0.0, diff_lim)),
        (np.angle(as_aligned), "ASM exit phase", "twilight", (-np.pi, np.pi)),
        (np.angle(wpm), "WPM exit phase", "twilight", (-np.pi, np.pi)),
        (phase_diff, "phase(WPM) - phase(ASM)", "coolwarm", (-np.pi, np.pi)),
    ]

    for ax, (arr, title, cmap, limits) in zip(axes.ravel(), panels):
        im = ax.imshow(arr, cmap=cmap, vmin=limits[0], vmax=limits[1])
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        add_colorbar(fig, ax, im)

    fig.suptitle("Exit-wave comparison after one Au [100] visual unit cell", fontsize=12)
    fig.tight_layout()
    return fig


plot_exit_comparison(exit_as, exit_wpm)
plt.show()


## Diffraction-Space View

The method gap is often easiest to interpret in Fourier space. WPM changes how high transverse-frequency components accumulate phase inside the medium, so the difference tends to live away from the central beam when the example is sufficiently high angle.


In [ ]:
def plot_diffraction_comparison(exit_as, exit_wpm, sampling, energy):
    dp_as = diffraction_intensity(exit_as)
    dp_wpm = diffraction_intensity(exit_wpm)
    abs_diff = np.abs(dp_wpm - dp_as)

    positive = np.concatenate([dp_as[dp_as > 0], dp_wpm[dp_wpm > 0]])
    vmax = max(np.percentile(positive, 99.9), 1e-18)
    vmin = max(vmax * 1e-8, 1e-20)
    diff_vmax = max(np.percentile(abs_diff[abs_diff > 0], 99.5) if np.any(abs_diff > 0) else 1e-18, 1e-18)

    theta = angle_grid_mrad(dp_as.shape, sampling, energy)
    centers, ring_gap = annular_relative_l2(dp_as, dp_wpm, theta)

    fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
    panels = [
        (dp_as, "ASM diffraction", "magma", LogNorm(vmin=vmin, vmax=vmax)),
        (dp_wpm, "WPM diffraction", "magma", LogNorm(vmin=vmin, vmax=vmax)),
        (abs_diff, "abs(WPM - ASM)", "inferno", LogNorm(vmin=max(diff_vmax * 1e-6, 1e-20), vmax=diff_vmax)),
    ]

    for ax, (arr, title, cmap, norm) in zip(axes[:3], panels):
        im = ax.imshow(arr, cmap=cmap, norm=norm)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        add_colorbar(fig, ax, im)

    axes[3].semilogy(centers, ring_gap, "o-", ms=3)
    axes[3].set_xlabel("scattering angle (mrad)")
    axes[3].set_ylabel("annular relative L2")
    axes[3].set_title("where the gap lives", fontsize=9)
    axes[3].grid(True, which="both", alpha=0.3)

    fig.suptitle("Diffraction-space comparison for the Au [100] visual sample", fontsize=12)
    fig.tight_layout()
    return fig


plot_diffraction_comparison(exit_as, exit_wpm, sampling, energy)
plt.show()


## Optional Bin-Count Check

The Au map is continuous, so WPM interpolation should become less visually blocky as the number of refractive-index bins increases. This is a diagnostic view only; it is not a convergence study for the full paper simulations.


In [ ]:
diagnostic_fine = wpm_diagnostic_step(
    probe, n_map, dz, energy, sampling, n_bins=16, power_spacing=1.0
)

print("16-bin WPM reference bins, reported as 1e3 x (n - 1):")
print(np.array2string(1e3 * (as_numpy(diagnostic_fine["n_refs"]) - 1.0), precision=4))
print(
    "weight range: "
    f"{as_numpy(diagnostic_fine['weights']).min():.3f} to "
    f"{as_numpy(diagnostic_fine['weights']).max():.3f}"
)

plot_wpm_diagnostics(n_map, diagnostic_fine, title_suffix=" (16 bins)")
plt.show()


## Sanity Checks

These checks are small and deterministic. They make sure the notebook examples agree with the expected limiting cases and that the Au sample is actually non-uniform.


In [ ]:
check_shape = (64, 64)
check_sampling = (0.15, 0.15)
check_probe = make_tilted_gaussian_probe(check_shape, check_sampling, energy, waist_A=1.8, tilt_x_mrad=60.0)

vacuum_n = jnp.ones(check_shape, dtype=jnp.float64)
as_vac = asm_step_local_n(check_probe, vacuum_n, dz, energy, check_sampling)
wpm_vac, _, _, _ = wpm_step_adaptive(check_probe, vacuum_n, dz, energy, check_sampling, n_bins=2, power_spacing=1.0)
vacuum_error = relative_l2(as_vac, wpm_vac)
assert vacuum_error < 1e-9, f"Vacuum ASM/WPM mismatch: {vacuum_error:.3e}"

uniform_n_value = float(as_numpy(n_map).mean())
uniform_n = uniform_n_value * jnp.ones(check_shape, dtype=jnp.float64)
wpm_uniform, _, _, _ = wpm_step_adaptive(check_probe, uniform_n, dz, energy, check_sampling, n_bins=2, power_spacing=1.0)
expected_uniform = homogeneous_wpm_field(check_probe, uniform_n_value, dz, energy, check_sampling)
uniform_error = relative_l2(wpm_uniform, expected_uniform)
assert uniform_error < 1e-9, f"Uniform-medium WPM mismatch: {uniform_error:.3e}"

gold_n = as_numpy(n_map)
assert gold_n.max() > gold_n.min(), "Au visual sample should have refractive-index contrast."
assert gold_n.min() >= 1.0, "Positive electrostatic Au potential should produce n >= 1."

weights_np = as_numpy(diagnostic["weights"])
assert np.all(weights_np >= -1e-12) and np.all(weights_np <= 1.0 + 1e-12), "WPM weights must stay in [0, 1]."

print(f"Vacuum ASM/WPM relative L2: {vacuum_error:.2e}")
print(f"Uniform WPM/expected homogeneous relative L2: {uniform_error:.2e}")
print(f"Au n contrast, max-min: {gold_n.max() - gold_n.min():.3e}")
print("WPM interpolation weights stay in [0, 1].")


## Takeaways

- The explainer uses an Au [100] sample geometry, matching the gold examples elsewhere in the project.
- WPM reduces to the angular-spectrum behavior in vacuum and to a single homogeneous angular-spectrum propagation in a uniform medium.
- WPM is more expensive than ASM because it evaluates a bank of homogeneous propagators, one per refractive-index bin, then recombines them spatially.
- WPM can improve on ASM when propagation should occur inside a laterally varying refractive-index medium rather than through vacuum after a phase screen.
- The visual potential gain is only there to make the small CPU example readable; the sample identity remains Au [100].
